# Updated ML Pipeline with Modular Models

This notebook shows how to use the new modular models structure with hyperparameter tuning.

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
np.random.seed(42)

import os
from glob import glob
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Import our new models structure
from models import ModelFactory, tune_all_models, compare_tuned_vs_untuned

## Data Loading

In [2]:
FILE = "data/mumbai_data_processed.csv"
USE_COLS = ['year','month','WS','WD','SP','BLH','TCC','T2M','D2M','RH','SSR','TP', 'PM25_WUstl']
df = pd.read_csv(FILE)

df.loc[df['year'] == 2024, 'PM25_WUstl'] = df.loc[df['year'] == 2024, 'PM25_clean']
df = df[USE_COLS]
print(df.info())

TARGET = 'PM25_WUstl'  
X = df.drop(columns=TARGET)
y = df[TARGET]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 324 entries, 0 to 323
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   year        324 non-null    int64  
 1   month       324 non-null    int64  
 2   WS          324 non-null    float64
 3   WD          324 non-null    float64
 4   SP          324 non-null    float64
 5   BLH         324 non-null    float64
 6   TCC         324 non-null    float64
 7   T2M         324 non-null    float64
 8   D2M         324 non-null    float64
 9   RH          324 non-null    float64
 10  SSR         324 non-null    float64
 11  TP          324 non-null    float64
 12  PM25_WUstl  324 non-null    float64
dtypes: float64(11), int64(2)
memory usage: 33.0 KB
None


## Transformation Pipeline

In [3]:
from scripts import imputer
from joblib import Memory
memory = Memory(location='cache_folder', verbose=0)

In [4]:
# Define column types
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

In [5]:
# Preprocessing steps
numeric_transformer = Pipeline(steps=[
    ('seasonal_imputer', imputer.SeasonalRegressorImputer(time_col='month')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Column processor pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

## Data Split

In [6]:
# Data split
train_mask = df["year"] < 2024
test_mask = df["year"] == 2024

X_train = df.loc[train_mask].drop(columns=["PM25_WUstl"])
y_train = df.loc[train_mask, "PM25_WUstl"]
X_test = df.loc[test_mask].drop(columns=["PM25_WUstl"])
y_test = df.loc[test_mask, "PM25_WUstl"]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (312, 12)
X_test shape: (12, 12)
y_train shape: (312,)
y_test shape: (12,)


## NEW: Using Modular Models Structure

### Option 1: Use models without hyperparameter tuning (like before)

In [7]:
# Create models using the factory (equivalent to the old approach)
models = ModelFactory.create_all_models(random_state=42)

print("Available models:")
for name in models.keys():
    print(f"  - {name}")

Available models:
  - Random Forest
  - LightGBM
  - Ridge Regression
  - Lasso Regression
  - Decision Tree
  - ExtraTreeRegressor
  - XGBoost
  - HistGradientBoosting
  - SVR
  - KNN
  - GAM
  - MLPRegressor
  - ANN
  - RNN
  - LSTM
  - BI-LSTM
  - GRU


In [8]:
# Fit and evaluate (same as before)
actual_pred = []
results = []

# Store actual values as a column for DataFrame
actual_dict = {"Actual": y_test.values}

# For each model, fit, predict, and collect results
for name, model in models.items():
    print(f"Training and evaluating: {name}")
    
    # Get untuned pipeline (equivalent to the old approach)
    pipeline = model.get_untuned_model(preprocessor)
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    # Add predictions to dict for DataFrame
    actual_dict[name] = y_pred

    results.append({
        "Model": name,
        "R²": round(r2, 4),
        "RMSE": round(rmse, 4),
        "MAE": round(mae, 4)
    })

# Display actual and predicted values as columns
display(pd.DataFrame(actual_dict))
display(pd.DataFrame(results))

Training and evaluating: Random Forest


Training and evaluating: LightGBM
Training and evaluating: Ridge Regression
Training and evaluating: Lasso Regression
Training and evaluating: Decision Tree
Training and evaluating: ExtraTreeRegressor


c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Training and evaluating: XGBoost
Training and evaluating: HistGradientBoosting
Training and evaluating: SVR
Training and evaluating: KNN
Training and evaluating: GAM
Training and evaluating: MLPRegressor


c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


Training and evaluating: ANN


c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


Training and evaluating: RNN


AttributeError: 'super' object has no attribute '__sklearn_tags__'

### Option 2: NEW - Hyperparameter Tuning

This is the new capability that wasn't available before!

In [ ]:
# Perform hyperparameter tuning for all models
# Note: This will take some time depending on n_iter and cv settings
print("Starting hyperparameter tuning...")
print("This may take several minutes depending on your settings.")

tuning_results = tune_all_models(
    X_train=X_train,
    y_train=y_train,
    preprocessor=preprocessor,
    models_to_tune=None,  # Tune all models
    n_iter=20,  # Reduced for faster execution
    cv=3,      # Reduced for faster execution
    n_jobs=-1,
    random_state=42
)

Starting hyperparameter tuning...
This may take several minutes depending on your settings.
Starting hyperparameter tuning for 10 models...
Models to tune: ['Random Forest', 'LightGBM', 'Ridge Regression', 'Lasso Regression', 'Decision Tree', 'ExtraTreeRegressor', 'XGBoost', 'HistGradientBoosting', 'SVR', 'KNN']
Number of iterations: 20
Cross-validation folds: 3
--------------------------------------------------

Tuning Random Forest...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
12 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
12 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\chand\AppDat

Best parameters: {'regressor__oob_score': True, 'regressor__n_estimators': 50, 'regressor__min_samples_split': 20, 'regressor__min_samples_leaf': 4, 'regressor__max_features': 'sqrt', 'regressor__max_depth': 10, 'regressor__criterion': 'absolute_error', 'regressor__bootstrap': True}
Best RMSE score: 14.5422
✓ Random Forest tuning completed successfully!

Tuning LightGBM...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best parameters: {'regressor__subsample': 0.6, 'regressor__reg_lambda': 0, 'regressor__reg_alpha': 10, 'regressor__objective': 'regression_l1', 'regressor__num_leaves': 50, 'regressor__n_estimators': 300, 'regressor__min_child_samples': 10, 'regressor__max_depth': 13, 'regressor__learning_rate': 0.1, 'regressor__colsample_bytree': 1.0, 'regressor__boosting_type': 'gbdt'}
Best RMSE score: 12.8725
✓ LightGBM tuning completed successfully!

Tuning Ridge Regression...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best parameters: {'regressor__sol

c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.380e+02, tolerance: 2.144e+01
  model = cd_fast.enet_coordinate_descent(
c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
9 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
4 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\sit

Best parameters: {'regressor__selection': 'cyclic', 'regressor__fit_intercept': True, 'regressor__alpha': 0.01}
Best RMSE score: 14.9547
✓ Lasso Regression tuning completed successfully!

Tuning Decision Tree...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best parameters: {'regressor__min_samples_split': 10, 'regressor__min_samples_leaf': 4, 'regressor__max_features': 'sqrt', 'regressor__max_depth': 7, 'regressor__criterion': 'absolute_error'}
Best RMSE score: 14.6227
✓ Decision Tree tuning completed successfully!

Tuning ExtraTreeRegressor...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
9 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
4 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\chand\AppData\

Best parameters: {'regressor__min_samples_split': 2, 'regressor__min_samples_leaf': 2, 'regressor__max_features': None, 'regressor__max_depth': 5, 'regressor__criterion': 'poisson'}
Best RMSE score: 14.8316
✓ ExtraTreeRegressor tuning completed successfully!

Tuning XGBoost...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best parameters: {'regressor__subsample': 1.0, 'regressor__reg_lambda': 0.1, 'regressor__reg_alpha': 1, 'regressor__n_estimators': 100, 'regressor__max_depth': 3, 'regressor__learning_rate': 0.05, 'regressor__gamma': 0, 'regressor__colsample_bytree': 0.6}
Best RMSE score: 14.6372
✓ XGBoost tuning completed successfully!

Tuning HistGradientBoosting...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
27 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
27 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\chand\AppDat

Best parameters: {'regressor__min_samples_leaf': 50, 'regressor__max_iter': 100, 'regressor__max_depth': 3, 'regressor__max_bins': 255, 'regressor__learning_rate': 0.2, 'regressor__l2_regularization': 0}
Best RMSE score: 14.0361
✓ HistGradientBoosting tuning completed successfully!

Tuning SVR...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best parameters: {'regressor__kernel': 'rbf', 'regressor__gamma': 'auto', 'regressor__epsilon': 0.2, 'regressor__degree': 4, 'regressor__C': 100}
Best RMSE score: 10.9006
✓ SVR tuning completed successfully!

Tuning KNN...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best parameters: {'regressor__weights': 'uniform', 'regressor__p': 2, 'regressor__n_neighbors': 9, 'regressor__leaf_size': 50, 'regressor__algorithm': 'auto'}
Best RMSE score: 12.7913
✓ KNN tuning completed successfully!

HYPERPARAMETER TUNING SUMMARY
Random Forest:
  Best RMSE: 14.5422
  Best params: {'regressor__oob_score': True, 'regressor__n_estimator

In [ ]:
tuning_results

{'Random Forest': {'model': <models.random_forest_model.RandomForestModel at 0x1e6970a17f0>,
  'best_params': {'regressor__oob_score': True,
   'regressor__n_estimators': 50,
   'regressor__min_samples_split': 20,
   'regressor__min_samples_leaf': 4,
   'regressor__max_features': 'sqrt',
   'regressor__max_depth': 10,
   'regressor__criterion': 'absolute_error',
   'regressor__bootstrap': True},
  'best_score': 14.542189625110767,
  'best_estimator': Pipeline(steps=[('preprocessor',
                   ColumnTransformer(transformers=[('num',
                                                    Pipeline(steps=[('seasonal_imputer',
                                                                     SeasonalRegressorImputer()),
                                                                    ('scaler',
                                                                     StandardScaler())]),
                                                    ['year', 'month', 'WS', 'WD',
               

### Option 3: NEW - Compare Tuned vs Untuned Performance

In [ ]:
comparison_df = compare_tuned_vs_untuned(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    preprocessor=preprocessor,
    tuning_results=tuning_results
)

print("Model Performance Comparison (Tuned vs Untuned):")
display(comparison_df)

c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.380e+02, tolerance: 2.144e+01
  model = cd_fast.enet_coordinate_descent(


Model Performance Comparison (Tuned vs Untuned):


,Model,Untuned_R2,Tuned_R2,R2_Improvement(%),Untuned_RMSE,Tuned_RMSE,RMSE_Improvement(%),Untuned_MAE,Tuned_MAE,MAE_Improvement(%)
0,Random Forest,0.0661,0.2633,298.63,16.9327,15.0384,11.19,12.7711,11.2477,11.93
1,LightGBM,0.0698,0.6187,786.93,16.8992,10.8200,35.97,11.8351,8.7192,26.33
2,Ridge Regression,-533.9353,-425.2478,20.36,405.2450,361.7413,10.74,236.2368,218.8629,7.35
3,Lasso Regression,-0.1119,-481.5126,-430163.18,18.4758,384.8765,-1983.14,16.0591,228.0956,-1320.35
4,Decision Tree,0.0332,0.4660,1304.12,17.2281,12.8042,25.68,12.7336,9.1509,28.14
5,ExtraTreeRegressor,0.0413,0.7710,1765.44,17.1554,8.3852,51.12,12.2197,7.0340,42.44
6,XGBoost,0.2670,-0.0834,-131.25,15.0012,18.2377,-21.58,12.2392,13.5453,-10.67
7,HistGradientBoosting,0.0206,0.0579,181.74,17.3403,17.0062,1.93,12.3998,11.3732,8.28
8,SVR,0.2018,-0.2130,-205.57,15.6543,19.2974,-23.27,12.7255,17.1938,-35.11
9,KNN,0.7096,0.6441,-9.24,9.4417,10.4530,-10.71,8.1100,9.0024,-11.00


### Option 4: NEW - Use Tuned Models for Final Evaluation

In [ ]:
# Evaluate all models (tuned if available, otherwise untuned)
final_results = []
actual_dict = {"Actual": y_test.values}

for name, model in models.items():
    print(f"Evaluating {name}...")
    model_type = "Untuned"
    pipeline = None
    # Use tuned model if available and tuning was successful
    if name in tuning_results and "error" not in tuning_results[name]:
        best_params = tuning_results[name].get('best_params')
        if best_params is not None:
            try:
                pipeline = model.get_tuned_model(preprocessor, best_params=best_params)
                model_type = "Tuned"
            except Exception as e:
                print(f"  [Warning] Could not use tuned parameters for {name}: {e}. Using untuned model.")
    # Fallback to untuned model if tuning not available or failed
    if pipeline is None:
        pipeline = model.get_untuned_model(preprocessor)
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    # Add predictions to dict for DataFrame
    actual_dict[f"{name} ({model_type})"] = y_pred
    
    final_results.append({
        "Model": f"{name} ({model_type})",
        "R²": round(r2, 4),
        "RMSE": round(rmse, 4),
        "MAE": round(mae, 4)
    })

# Display results
print("\nFinal Model Performance:")
display(pd.DataFrame(actual_dict))
display(pd.DataFrame(final_results))


Evaluating Random Forest...
Evaluating LightGBM...
Evaluating Ridge Regression...
Evaluating Lasso Regression...


c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\chand\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.380e+02, tolerance: 2.144e+01
  model = cd_fast.enet_coordinate_descent(


Evaluating Decision Tree...
Evaluating ExtraTreeRegressor...
Evaluating XGBoost...
Evaluating HistGradientBoosting...
Evaluating SVR...
Evaluating KNN...

Final Model Performance:


,Actual,Random Forest (Tuned),LightGBM (Tuned),Ridge Regression (Tuned),Lasso Regression (Tuned),Decision Tree (Tuned),ExtraTreeRegressor (Tuned),XGBoost (Tuned),HistGradientBoosting (Tuned),SVR (Tuned),KNN (Tuned)
0,48.695304,83.086732,70.167278,44.517316,51.109548,84.746760,50.533190,92.850754,85.693367,33.443620,66.241373
1,47.682459,75.815038,67.851668,31.246647,38.283165,49.685068,50.533190,75.528236,80.723415,30.197723,58.301556
2,35.903431,54.653106,51.509091,7.667499,16.368448,49.685068,53.547350,59.695038,60.863570,33.093239,49.685951
3,32.798450,42.015864,38.195926,-2.609918,3.414636,29.138395,32.360691,40.292419,35.266350,35.819463,31.738491
4,26.107197,21.562717,18.756098,-23.153632,-19.380576,18.591808,19.815375,19.951349,12.845469,37.117516,24.875256
5,14.760085,19.438720,18.701389,-430.207832,-457.546995,18.591808,19.815375,19.486582,13.352666,39.566502,19.762855
6,12.651744,21.290162,21.781180,-973.132994,-1040.181501,24.526024,19.815375,24.906462,18.848372,39.566502,20.462476
7,12.615327,18.961268,18.785614,-457.920864,-486.698205,18.591808,19.815375,19.486582,13.351568,39.566502,19.762855
8,15.421420,19.698713,18.822213,-375.586515,-398.335513,18.591808,19.815375,19.384014,12.255664,39.566502,19.644482
9,31.320013,31.230335,32.413043,-107.910883,-113.071468,21.989335,19.815375,33.333309,29.504404,39.566502,19.033485


,Model,R²,RMSE,MAE
0,Random Forest (Tuned),0.2633,15.0384,11.2477
1,LightGBM (Tuned),0.6187,10.8200,8.7192
2,Ridge Regression (Tuned),-425.2478,361.7413,218.8629
3,Lasso Regression (Tuned),-481.5126,384.8765,228.0956
4,Decision Tree (Tuned),0.4660,12.8042,9.1509
5,ExtraTreeRegressor (Tuned),0.7710,8.3852,7.0340
6,XGBoost (Tuned),-0.0834,18.2377,13.5453
7,HistGradientBoosting (Tuned),0.0579,17.0062,11.3732
8,SVR (Tuned),-0.2130,19.2974,17.1938
9,KNN (Tuned),0.6441,10.4530,9.0024


Ridge Regression
Lasso Regression
ElasticNet
Decision Tree
XGBoost
HistGradientBoosting
MLPRegressor
SVR
KNN
CERT 
GAM (Generative additive model)

--------

## Summary of Changes

### What Changed:
1. **Before**: Models were defined as simple dictionaries with sklearn estimators
2. **After**: Models are created using a factory pattern with additional capabilities

### What's New:
1. **Hyperparameter Tuning**: Built-in RandomizedSearchCV for all models
2. **Performance Comparison**: Easy comparison between tuned and untuned models
3. **Modular Structure**: Each model is in its own file, easy to extend
4. **Factory Pattern**: Easy model creation and management

### Backward Compatibility:
- The basic usage (Option 1) works exactly like before
- You can still use the same workflow if you don't want hyperparameter tuning
- All existing code will continue to work with minimal changes